In [4]:
# ============================================================
# TWO-STAGE HYBRID CLASSIFIER — FINAL VERSION (STRATEGY C)
# ============================================================

import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ============================================================
# CONFIG
# ============================================================
DEV_PATH  = "../data/raw/development.csv"
EVAL_PATH = "../data/raw/evaluation.csv"
SUB_PATH  = "submission.csv"

MIN_RULE_SUPPORT = 30
MIN_RULE_PURITY  = 0.95
RULE_PRIORITY    = "best_purity_then_freq"
C_VALUE          = 1.5

# ============================================================
# LOAD DATA
# ============================================================
df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

# ============================================================
# TIMESTAMP PARSE
# ============================================================
for df in [df_dev, df_eval]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# DROP TIMESTAMP NaN **ONLY ON DEV**
df_dev = df_dev[df_dev["timestamp"].notna()].reset_index(drop=True)

print("DEV samples:", len(df_dev))
print("EVAL samples:", len(df_eval))

# ============================================================
# BASIC FIXES
# ============================================================
for df in [df_dev, df_eval]:
    df["article"] = df["article"].fillna("").astype(str)
    df["title"]   = df["title"].fillna("").astype(str)
    df["source"]  = df["source"].fillna("").astype(str)

# ============================================================
# TEXT
# ============================================================
def build_text(df):
    return (df["title"] + " " + df["article"]).str.lower()

df_dev["text"]  = build_text(df_dev)
df_eval["text"] = build_text(df_eval)

# ============================================================
# NUMERIC FEATURES
# ============================================================
def add_numeric(df):
    df["n_tokens"]    = df["article"].str.split().str.len()
    df["title_len"]   = df["title"].str.len()
    df["article_len"] = df["article"].str.len()
    df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
    df["year"]  = df["timestamp"].dt.year
    df["month"] = df["timestamp"].dt.month
    df["dow"]   = df["timestamp"].dt.dayofweek
    return df

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = [
    "n_tokens", "title_len", "article_len",
    "title_ratio", "year", "month", "dow"
]

# IMPORTANT: fill NaN numerics (EVAL SAFE)
for df in [df_dev, df_eval]:
    df[NUM_COLS] = df[NUM_COLS].fillna(0)

# ============================================================
# FEATURES
# ============================================================
FEATURES = [
    "source", "text",
    "n_tokens", "title_len", "article_len",
    "title_ratio", "year", "month", "dow"
]

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]

# ============================================================
# RULE MINING
# ============================================================
def tokenize_for_rules(text):
    return text.split()

def mine_pure_rules(texts, labels):
    counts = defaultdict(lambda: Counter())

    for txt, y in zip(texts, labels):
        for tok in set(tokenize_for_rules(txt)):
            counts[tok][y] += 1

    rule_token_to_class = {}
    rule_meta = {}

    for tok, c in counts.items():
        total = sum(c.values())
        if total < MIN_RULE_SUPPORT:
            continue

        best_class, best_freq = c.most_common(1)[0]
        purity = best_freq / total

        if purity >= MIN_RULE_PURITY:
            rule_token_to_class[tok] = best_class
            rule_meta[tok] = (purity, total)

    return rule_token_to_class, rule_meta

def apply_rules(texts, rule_token_to_class, rule_meta):
    rule_pred = np.full(len(texts), -1, dtype=int)

    for i, txt in enumerate(texts):
        toks = set(tokenize_for_rules(txt))
        hits = [t for t in toks if t in rule_token_to_class]
        if not hits:
            continue

        if RULE_PRIORITY == "best_purity_then_freq":
            hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)
        else:
            hits.sort(key=lambda t: rule_meta[t][1], reverse=True)

        rule_pred[i] = rule_token_to_class[hits[0]]

    return rule_pred

# ============================================================
# MODEL
# ============================================================
def make_model():
    pre = ColumnTransformer(
        transformers=[
            ("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

            ("w_tfidf", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1,2),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=250_000
            ), "text"),

            ("c_tfidf", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3,5),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=300_000
            ), "text"),

            ("num", StandardScaler(), NUM_COLS)
        ],
        remainder="drop",
        n_jobs=-1
    )

    clf = LogisticRegression(
        C=C_VALUE,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    )

    return Pipeline([
        ("pre", pre),
        ("clf", clf)
    ])

# ============================================================
# TRAIN ON FULL DEV
# ============================================================
print("\nTraining model on full development...")
model = make_model()
model.fit(X_dev, y_dev)

print("Mining rules on full development...")
rule_token_to_class, rule_meta = mine_pure_rules(df_dev["text"], y_dev)
print("Total rules:", len(rule_token_to_class))

# ============================================================
# PREDICT ON EVAL
# ============================================================
print("Predicting on evaluation...")
model_pred = model.predict(X_eval)
rule_pred  = apply_rules(df_eval["text"], rule_token_to_class, rule_meta)

final_pred = model_pred.copy()
mask = rule_pred != -1
final_pred[mask] = rule_pred[mask]

print(f"Rule coverage on eval: {mask.mean():.3f}")

# ============================================================
# SUBMISSION
# ============================================================
submission = pd.DataFrame({
    "Id": df_eval["Id"],
    "label": final_pred
})

submission.to_csv(SUB_PATH, index=False)
print("Submission saved to:", SUB_PATH)



DEV samples: 52247
EVAL samples: 20000

Training model on full development...
Mining rules on full development...
Total rules: 94
Predicting on evaluation...
Rule coverage on eval: 0.057
Submission saved to: submission.csv
